In [1]:
# @title
!pip install chromadb
!pip install colab-xterm
!pip install pypdf
!pip install torch
!pip install langchain_community
!pip install langchain_classic
!pip install langchain_huggingface
!pip install langchain_ollama

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling openteleme

In [3]:
#Task 1: Document Loading and Preprocessing

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

loader = PyPDFLoader('HR Policy Manual 2023.pdf')
documents = loader.load()

def preprocess_documents(documents):
  for doc in documents:
    doc.page_content = re.sub(r'[^a-zA-Z0-9 \n.]', '', doc.page_content)
    doc.page_content = re.sub(r'\.+', '.', doc.page_content)
  return documents

def split_documents(documents):
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,
      chunk_overlap=100)
  documents = text_splitter.split_documents(documents)
  return documents

documents = preprocess_documents(documents)
documents = split_documents(documents)

In [4]:
for document in documents[25:29]:
    print(f'Chunk text - {document.page_content}\n')

Chunk text - Contribution by the Institute .174
Interest .174
Advance from the Fund .175
Withdrawals from the Fund . 177
Circumstances in which Accumulations are payable .179
Procedure .181
Gratuity .182
B National Pension System . 184
Chapter 23 Retention of Documents .196

Chunk text - 1
IIMA HR Policy Manual 2023
IIMA BRIEF NOTE
Indian Institute of Management Ahmedabad IIMA was set up by the Government of India in 
collaboration with the Government of Gujarat and local industrialists as an autonomous Institute 
in 1961. IIMA has been conceived not only as a business school but also as a management institute. 
IIMA builds on over five decades of excellence and leadership in management education.

Chunk text - IIMA builds on over five decades of excellence and leadership in management education.
IIMA has been rated as Indias best and Asias foremost Business School. IIMA continues to be 
ranked as one of the finest institutions in the world in management education with an academic 
rig

In [5]:
#Task 2: Text Embedding and Vector Store Setup - Using langchain libraries
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
import chromadb

# Use a valid HuggingFace model
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
vdb = chromadb.PersistentClient(path='/content/chroma_db')

# Pass the function directly
db = Chroma.from_documents(
    documents=documents,
    embedding=embedding_function,
    client = vdb,
    collection_name='hr_policy_manual',
)

/tmp/ipykernel_10264/1983234359.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [17]:
query_list = ['What is IIMA', 'What are the major programmes offered by IIMA?']
for query in query_list:
  results = db.similarity_search(query, k=2)
  for doc in results:
    print(f'Results for query {query} - {doc.page_content}\n')

Results for query What is IIMA - IIMA builds on over five decades of excellence and leadership in management education.
IIMA has been rated as Indias best and Asias foremost Business School. IIMA continues to be 
ranked as one of the finest institutions in the world in management education with an academic 
rigour that matches the top league. With a distinguished faculty an exceptional studentfaculty 
ratio and a 100acre worldclass campus conducive to continuous learning IIMA is an Institute

Results for query What is IIMA - 1
IIMA HR Policy Manual 2023
IIMA BRIEF NOTE
Indian Institute of Management Ahmedabad IIMA was set up by the Government of India in 
collaboration with the Government of Gujarat and local industrialists as an autonomous Institute 
in 1961. IIMA has been conceived not only as a business school but also as a management institute. 
IIMA builds on over five decades of excellence and leadership in management education.

Results for query What are the major programmes of

In [9]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [10]:
!nohup ollama serve &

nohup: appending output to 'nohup.out'


In [12]:
!ollama -v

ollama version is 0.20.2


In [11]:
!ollama pull llama3.2
!ollama list


NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    


In [35]:
from langchain_ollama.llms import OllamaLLM
import torch
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate

# Define custom templates
condense_template = "Given the following conversation history and a follow-up user question, \
  rephrase the follow-up question to be a standalone question that can be understood without \
  the chat history. Do not answer the question, just rephrase it.\
  Chat History:\
    {chat_history}\
  Follow-up Question:\
    {question}\
  Standalone Question:"

qa_template = "You are an assistant for question-answering tasks. \
    Use the following pieces of retrieved context and the provided chat history to answer \
    the question. If the answer is in the context, use it. If the answer is not in the context,\
    use your internal knowledge. If you don't know the answer, say you don't know.\
    Keep the answer concise and conversational.\
    Chat History: \
      {chat_history}\
    Retrieved Context: \
      {context} \
    Question: \
      {question} \
    Answer:"

CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(condense_template)
QA_PROMPT = PromptTemplate.from_template(qa_template)

def get_model():
  # Initialize LLM
  # Detect GPU availability using torch
  if torch.cuda.is_available():
      device = torch.device("cuda")
      print('GPU detected. Using GPU for inference')
  else:
      device = torch.device("cpu")
      print('No GPU detected. Using CPU for inference')
  llm = OllamaLLM(base_url="http://127.0.0.1:11434", model="llama3.2", device=device)
  return llm

llm = get_model()
# Initialize memory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages= True)
# Create retriever from the ChromaDB instance 'db'
retriever = db.as_retriever()

# Create the ConversationalRetrievalChain
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    condense_question_prompt=CONDENSE_QUESTION_PROMPT,
    combine_docs_chain_kwargs={"prompt": QA_PROMPT},
    verbose=False
)

def run_rag_pipeline(qa_chain):
    print("Welcome! Ask a question (or type 'exit' to quit):")
    # chat_history is managed internally by ConversationBufferMemory attached to qa_chain
    while True:
        # 1. Ask for user query
        query = input("\nUser Query: ")

        # 2. Exit condition
        if query.lower() in ['exit', 'quit']:
            print("Exiting RAG system.")
            break

        # 3. RAG Processing (Retrieve -> Generate)
        # Pass only the question, memory handles history
        response = qa_chain.invoke({"question": query})

        # 4. Display result
        print(f'RAG Response: {response["answer"]}')

GPU detected. Using GPU for inference


In [36]:
query_list = ['What are the major programmes offered by IIMA?',
              'Tell me more about the two-year Post Graduate Programme in Management.' ]

run_rag_pipeline(qa_chain)

Welcome! Ask a question (or type 'exit' to quit):

User Query: Hello I am SG
RAG Response: Hi SG, welcome! I don't see much context about you in our chat history, so I'll need a bit more information to help. Can you tell me what's on your mind or how I can assist you today?

User Query: What is IIMA
RAG Response: Hi SG, I think I have a good idea what you're asking! Based on our conversation history, IIMA (Indian Institute of Management Ahmedabad) appears to be a prestigious business school and management institute in India. It's one of the top-ranked institutions globally, known for its excellence in management education, research, and teaching. Does that sound about right to you?

User Query: What are the major programs offered here
RAG Response: Hi SG, so you're interested in knowing about the academic programs offered by IIMA (Indian Institute of Management Ahmedabad). Based on what we've discussed earlier, it seems that IIMA offers a range of postgraduate programs. According to th

### Explanation of Document Loading and Preprocessing

In this section, we load our HR policy manual PDF and preprocess the text to make it suitable for embedding and retrieval.

1.  **Document Loading (`PyPDFLoader`):**
    *   **What it does:** The `PyPDFLoader` from `langchain_community.document_loaders` is used to extract text content from the `HR Policy Manual 2023.pdf` file.
    *   **Why:** This converts the unstructured PDF data into a format that can be processed (a list of `Document` objects, where each object typically represents a page from the PDF).

2.  **Preprocessing (`preprocess_documents` function):**
    *   **What it does:**
        *   `re.sub(r'[^a-zA-Z0-9 \n.]', '', doc.page_content)`: This regular expression removes any characters that are not alphanumeric, spaces, newlines, or periods. This helps in cleaning up special characters, symbols, and formatting artifacts that might confuse the embedding model or retrieval process.
        *   `re.sub(r'\.+', '.', doc.page_content)`: This collapses multiple consecutive periods into a single period. This standardizes punctuation and can prevent issues where multiple periods might be interpreted as separate sentences or noise.
    *   **Why:** Cleaning the text before embedding is crucial. Noise (like extra symbols or inconsistent punctuation) can lead to less accurate embeddings, meaning that semantically similar pieces of text might not be mapped close together in the vector space. This step enhances the quality of the embeddings and thus the retrieval accuracy.

3.  **Document Splitting (`RecursiveCharacterTextSplitter`):**
    *   **What it does:** The `RecursiveCharacterTextSplitter` divides the longer `Document` objects (e.g., full PDF pages) into smaller, overlapping chunks.
        *   `chunk_size=500`: Each chunk will aim to be around 500 characters long.
        *   `chunk_overlap=100`: Consecutive chunks will share 100 characters. This overlap helps ensure that context isn't lost at the boundaries between chunks when a relevant piece of information might be split across two chunks.
    *   **Why:** Large documents are impractical for direct use with LLMs and can dilute the relevance of embeddings. Splitting them into smaller, manageable chunks allows for more precise retrieval of relevant information. The overlap helps maintain continuity and ensures that the LLM has enough surrounding context when retrieving a chunk.

### Explanation of Embedding and Vector Search Setup

In this section, we set up the core components for a Retrieval-Augmented Generation (RAG) system: text embedding and a vector database.

1.  **Text Embedding (`SentenceTransformerEmbeddings`):**
    *   **What it does:** Text embedding converts human-readable text into numerical vectors (lists of numbers). The key is that these vectors capture the semantic meaning of the text, meaning that texts with similar meanings will have vectors that are numerically close to each other in a multi-dimensional space.
    *   **Model:** We use `all-MiniLM-L6-v2` from HuggingFace, a pre-trained model known for its efficiency and good performance in generating sentence embeddings.

2.  **Vector Store (`Chroma` and `Chroma.from_documents`):**
    *   **What it does:** A vector store is a database designed to efficiently store and search these numerical vectors. When you have a query, it's also converted into a vector, and then the vector store finds the most 'similar' vectors (and thus documents) to your query vector.
    *   **ChromaDB:** We initialize `Chroma` as our persistent vector database. `Chroma.from_documents` takes our preprocessed `documents` and the `embedding_function` to:
        *   Generate embeddings for each chunk of text in our documents.
        *   Store these embeddings, along with the original text, in the `hr_policy_manual` collection within our `chroma_db`.

### Conversational Memory for Context-Awareness and Continuity

To enhance context-awareness and ensure continuity across multiple turns in a conversation, we utilize `ConversationBufferMemory` from `langchain_classic.memory`.

1.  **What it does:**
    *   **Stores Chat History:** The `ConversationBufferMemory` object (`memory`) is initialized to keep track of the conversation's dialogue. It stores both user inputs and AI responses.
    *   **`memory_key="chat_history"`:** This parameter specifies the key under which the conversation history will be stored and passed to the Language Model (LLM).
    *   **`return_messages=True`:** This ensures that the chat history is returned as a list of message objects, which is often the preferred format for LLMs.

2.  **How it enhances context-awareness:**
    *   **`ConversationalRetrievalChain` Integration:** The `memory` object is directly integrated into the `ConversationalRetrievalChain` when the `qa_chain` is created.
    *   **`condense_question_prompt`:** When a follow-up question is asked, the chain first uses the `condense_question_prompt` along with the `chat_history` (from memory) to rephrase the follow-up question into a standalone question. This ensures that even if the user says something like "What about its programs?", the system understands "its" refers to the topic discussed in the previous turn.
    *   **`qa_template`:** The `chat_history` is also passed to the `qa_template` during the final answer generation phase. This allows the LLM to consider the entire conversation flow, not just the current question and retrieved documents, leading to more coherent and relevant responses that build upon previous exchanges.

**In essence, conversational memory allows our RAG system to remember past interactions, interpret new queries within that broader context, and maintain a natural, flowing dialogue with the user.**